# Bài 6 · UCT độ sâu 1 với UCB1

Mỗi nước ở gốc được thử một lần. Sau đó, thuật toán chọn nhánh tối đa hóa

$$\bar u_i + \sqrt{\frac{2\ln t}{n_i}}.$$

$\bar u_i$ là utility trung bình ở nhánh $i$, $t$ là tổng lượt mô phỏng đã chạy và $n_i$ là số lượt của nhánh. Số hạng đầu ưu tiên nhánh có kết quả tốt (**exploitation**), số hạng sau khuyến khích thử nhánh còn ít mẫu (**exploration**). Sau N lượt, tác tử chọn nước được thăm nhiều nhất, ưu tiên chỉ số nhỏ hơn khi đồng hạng. Đây là phiên bản giữ cây ở độ sâu 1 như notebook tham chiếu; chính sách dưới các nút con vẫn đi ngẫu nhiên.

In [1]:
from random import Random
from lab05_monte_carlo import UCT_depth1, pmcs, ucb1_10_player, ucb1_100_player
from lab05_experiments import CASES

board = CASES[0][1]
for search in (pmcs, UCT_depth1):
    move, details = search(board, 100, rng=Random(52000), return_stats=True)
    print(search.__name__, "chọn", move)
    for action, info in details.items():
        print(f"  ô {action}: {info['visits']:3d} mẫu, trung bình {info['mean']:+.3f}")
    assert sum(info["visits"] for info in details.values()) == 100
    assert all(info["visits"] >= 1 for info in details.values())
    assert move in details

pmcs chọn 8
  ô 2:  20 mẫu, trung bình +0.950
  ô 5:  20 mẫu, trung bình +0.600
  ô 6:  20 mẫu, trung bình +0.800
  ô 7:  20 mẫu, trung bình +0.750
  ô 8:  20 mẫu, trung bình +1.000
UCT_depth1 chọn 8
  ô 2:  22 mẫu, trung bình +0.909
  ô 5:  13 mẫu, trung bình +0.692
  ô 6:  17 mẫu, trung bình +0.824
  ô 7:  19 mẫu, trung bình +0.842
  ô 8:  29 mẫu, trung bình +1.000


Hai cách dùng cùng 100 lượt mô phỏng. PMCS thử mỗi ô 20 lần. UCB1 cấp 29 lượt cho ô 8 có utility quan sát cao nhất và vẫn thử ô 5 có utility thấp hơn 13 lần.

## Kiểm thử độ ổn định

So sánh cùng 30 seed, cùng thế và cùng ngân sách tổng. Chạy một lần để làm nóng bộ nhớ đệm trước khi đo. Giá trị Minimax dùng làm chuẩn chất lượng. Tần suất chọn ô phản ánh độ ổn định của quyết định.

In [2]:
from collections import Counter
import random
from time import perf_counter

from lab05_experiments import CASES, benchmark, move_values, print_benchmark

def measure_cases(search, budgets, trials=30):
    measurements = {}
    for label, board, player in CASES:
        values = move_values(board, player)
        optimum = max(values.values())
        print(f"{label} — giá trị Minimax theo ô: {values}")
        measurements[label] = {}
        search(board, max(budgets), player, rng=random.Random(51999))
        for budget in budgets:
            generators = [random.Random(52000 + i) for i in range(trials)]
            start = perf_counter()
            choices = [search(board, budget, player, rng=rng) for rng in generators]
            ms = (perf_counter() - start) * 1000 / trials
            counts = dict(sorted(Counter(choices).items()))
            optimal = sum(count for move, count in counts.items() if values[move] == optimum)
            measurements[label][budget] = {"choices": counts, "optimal": optimal, "ms": ms}
            print(f"  N={budget:4}  ô:số lần={counts}  tối ưu={optimal}/{trials}  {ms:.4f} ms/nước")
    return measurements

trials = measure_cases(UCT_depth1, (10, 100))

X thắng ngay — giá trị Minimax theo ô: {2: 1, 5: 0, 6: 1, 7: 0, 8: 1}
  N=  10  ô:số lần={2: 24, 5: 3, 6: 2, 7: 1}  tối ưu=26/30  0.0842 ms/nước
  N= 100  ô:số lần={8: 30}  tối ưu=30/30  0.7232 ms/nước
X phải chặn — giá trị Minimax theo ô: {3: -1, 5: -1, 6: -1, 7: 0, 8: -1}
  N=  10  ô:số lần={3: 5, 5: 5, 6: 3, 7: 15, 8: 2}  tối ưu=15/30  0.0913 ms/nước
  N= 100  ô:số lần={3: 1, 5: 3, 6: 1, 7: 25}  tối ưu=25/30  0.8915 ms/nước
X bất lợi (giả định) — giá trị Minimax theo ô: {1: -1, 3: -1, 4: -1, 5: -1, 6: -1, 7: -1}
  N=  10  ô:số lần={1: 6, 3: 5, 4: 11, 5: 1, 6: 5, 7: 2}  tối ưu=30/30  0.0969 ms/nước
  N= 100  ô:số lần={4: 30}  tối ưu=30/30  1.0867 ms/nước
Bàn trống — giá trị Minimax theo ô: {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0}
  N=  10  ô:số lần={0: 18, 1: 6, 2: 4, 3: 1, 4: 1}  tối ưu=30/30  0.1414 ms/nước
  N= 100  ô:số lần={0: 2, 1: 1, 2: 6, 4: 12, 5: 1, 6: 4, 7: 1, 8: 3}  tối ưu=30/30  1.7751 ms/nước
X thua cưỡng bức — giá trị Minimax theo ô: {2: -1, 7: -1, 8: -1}

Ở ca X thắng ngay, UCB1 chọn ô thắng ngay 8 ở cả 30 lần với N=100. Ở ca X phải chặn, số lần chọn ô 7 tăng từ 15/30 lên 25/30. Trên bàn trống, N=10 chỉ còn một lượt phân bổ sau khi thử chín ô. Với N=100, các nước mở đầu vẫn phân tán giữa nhiều ô. Quyết định ổn định hơn khi kết quả mô phỏng giữa các nước có chênh lệch rõ.

## Đấu với Random

Chạy 100 ván ở mỗi vai cho mỗi ngân sách. Thời gian tính riêng lượt UCB1, bao gồm cập nhật chỉ số chọn nhánh.

In [3]:
stats = {}
for budget, agent in [(10, ucb1_10_player), (100, ucb1_100_player)]:
    random.seed(53000)
    stats[budget] = benchmark(agent)
    print_benchmark(f"UCB1 N={budget}", stats[budget])

UCB1 N=10
  Vai X: 85 thắng, 7 hòa, 8 thua / 100 ván; 0.2168 ms/nước
  Vai O: 68 thắng, 6 hòa, 26 thua / 100 ván; 0.1753 ms/nước
  Tổng: 153 thắng, 13 hòa, 34 thua / 200 ván; 0.1971 ms/nước
UCB1 N=100
  Vai X: 99 thắng, 0 hòa, 1 thua / 100 ván; 1.4090 ms/nước
  Vai O: 85 thắng, 4 hòa, 11 thua / 100 ván; 1.1243 ms/nước
  Tổng: 184 thắng, 4 hòa, 12 thua / 200 ván; 1.2708 ms/nước


Tăng N từ 10 lên 100 giảm số ván thua từ 34 xuống 12. PMCS và UCB1 cùng thắng 184/200 ván ở N=100, với lần lượt 9 và 12 ván thua. Phân bổ theo UCB1 giúp tập trung vào các nhánh hứa hẹn, nhưng kết quả còn phụ thuộc vào nhiễu ở các lượt thử đầu và tiêu chí chọn theo số lượt thăm.